In [50]:
import pandas as pd
import numpy as np
from sklearn import metrics 
from sklearn.metrics import confusion_matrix
from aif360 import metrics as fairmetrics
from aif360.datasets import BinaryLabelDataset
import seaborn as sns

In [22]:
full_df = pd.read_csv('adult_reconstruction_bin.csv')
models_df = pd.read_csv('adult_models_only.csv')

In [23]:
merged_df = pd.merge(full_df, models_df, on= 'Unnamed: 0')

*************
### Randomly Selected Models:

In [24]:
my_num = 80
np.random.seed(my_num)
models_to_audit = np.random.choice( models_df.columns,2)
models_to_audit

array(['GPR_income>=60k', 'LR_income>=60k'], dtype=object)

**************
### Audit of GPR_income>=60k:

In [108]:
GPR_accuracy = metrics.accuracy_score(merged_df['income>=60k'],
                       merged_df['GPR_income>=60k'])

In [109]:
GPR_precision = metrics.precision_score(merged_df['income>=60k'],
                       merged_df['GPR_income>=60k'])

In [110]:
GPR_recall = metrics.recall_score(merged_df['income>=60k'],
                       merged_df['GPR_income>=60k'])

In [115]:
GPR_data = [GPR_accuracy, GPR_precision, GPR_recall]
pd.DataFrame(data= GPR_data,
    columns=['Percentage'],
    index=['Accuracy Score', 'Precision Score',
             'Recall Score']
)

,Percentage
Accuracy Score,0.864302
Precision Score,0.705750
Recall Score,0.366511


The GPR model predicts whether a person earns greater than or equal to a certain income threshold. I examined the case of income ≥ 60k to evaluate how well the model could correctly identify individuals who make 60k or more. To do this, I looked at three metrics: Accuracy, Precision, and Recall.

Accuracy measures the percentage of all predictions that the model got right. A high accuracy score means the model is making the majority of its predictions correctly. Precision measures how many of the predictions labeled as positive were actually correct; a high precision means there are very few false positives (people predicted to earn ≥ 60k but who actually earn < 60k). Recall measures how many of the actual positive cases the model successfully found; a high recall score means the model rarely misses someone who actually earns ≥ 60k.

Looking at the metrics for this model, we can see the following scores:

* Accuracy: 86.43%
* Precision: 70.58%
* Recall: 36.65%

These scores show that the GPR model correctly predicted a majority of who did and did not make 60k or more; however, this does not mean the model is perfect. The overall goal of the model is to determine who makes 60k or more per year, so it is especially important that it correctly identifies individuals in that category (true positives). To assess performance more fully, we can look at precision and recall.

The precision of 70.58% indicates that most of the people predicted to earn ≥ 60k actually did, meaning the false positive rate is fairly low. However, the recall score of 36.65% is less promising. It shows that the model missed many true high earners. In other words, only about 36% of individuals who actually made ≥ 60k were correctly identified as doing so.

### Do Race or Gender Skew Metrics of the Model?

#### Accuracy:

In [27]:
race_cat = lambda d: metrics.accuracy_score(merged_df['income>=60k'],
                       merged_df['GPR_income>=60k'])
merged_df.groupby('race')[['income>=60k', 'GPR_income>=60k']].apply(race_cat)

race
Amer-Indian-Eskimo    0.864302
Asian-Pac-Islander    0.864302
Black                 0.864302
Other                 0.864302
White                 0.864302
dtype: float64

In [28]:
gender_cat = lambda d: metrics.accuracy_score(merged_df['income>=60k'],
                       merged_df['GPR_income>=60k'])
merged_df.groupby('gender')[['income>=60k', 'GPR_income>=60k']].apply(gender_cat)

gender
Female    0.864302
Male      0.864302
dtype: float64

#### Precision:

In [29]:
race_cat = lambda d: metrics.precision_score(merged_df['income>=60k'],
                       merged_df['GPR_income>=60k'])
merged_df.groupby('race')[['income>=60k', 'GPR_income>=60k']].apply(race_cat)

race
Amer-Indian-Eskimo    0.70575
Asian-Pac-Islander    0.70575
Black                 0.70575
Other                 0.70575
White                 0.70575
dtype: float64

In [30]:
gender_cat = lambda d: metrics.precision_score(merged_df['income>=60k'],
                       merged_df['GPR_income>=60k'])
merged_df.groupby('gender')[['income>=60k', 'GPR_income>=60k']].apply(gender_cat)

gender
Female    0.70575
Male      0.70575
dtype: float64

#### Recall:

In [103]:
race_cat = lambda d: metrics.recall_score(merged_df['income>=60k'],
                       merged_df['GPR_income>=60k'])
merged_df.groupby('race')[['income>=60k', 'GPR_income>=60k']].apply(race_cat)

race
Amer-Indian-Eskimo    0.366511
Asian-Pac-Islander    0.366511
Black                 0.366511
Other                 0.366511
White                 0.366511
dtype: float64

In [102]:
gender_cat = lambda d: metrics.recall_score(merged_df['income>=60k'],
                       merged_df['GPR_income>=60k'])
merged_df.groupby('gender')[['income>=60k', 'GPR_income>=60k']].apply(gender_cat)

gender
Female    0.366511
Male      0.366511
dtype: float64

When splitting the accuracy, precision, and recall scores by race and gender, we can see that there is no variance between groups for any of the three metrics. This means that the model performs equally well across different racial and gender groups, suggesting that it is fair in its predictions and does not favor one group over another.

**************

### Audit of LR_income>=60k:

In [117]:
LR_accuracy = metrics.accuracy_score(merged_df['income>=60k'],
                       merged_df['LR_income>=60k'])

In [118]:
LR_precision = metrics.precision_score(merged_df['income>=60k'],
                       merged_df['LR_income>=60k'])

In [119]:
LR_recall = metrics.recall_score(merged_df['income>=60k'],
                       merged_df['LR_income>=60k'])

In [120]:
LR_data = [LR_accuracy, LR_precision, LR_recall]
pd.DataFrame(data = LR_data,
    columns=['Percentage'],
    index=['Accuracy Score', 'Precision Score',
             'Recall Score']
)

,Percentage
Accuracy Score,0.857027
Precision Score,0.665912
Recall Score,0.344262


The LR model also predicts whether a person earns greater than or equal to a certain income threshold. I examined the case of income ≥ 60k to evaluate how well the model could correctly identify individuals who make 60k or more in comparison to the GPR model. To do this, I looked at three metrics: Accuracy, Precision, and Recall.

Looking at the metrics for this model, we can see the following scores:

* Accuracy: 85.70%
* Precision: 66.59%
* Recall: 34.43%

Overall, the LR model correctly classified most individuals in terms of whether they earned 60k or more, though its results show room for improvement. Since the main objective is to identify higher income individuals, correctly predicting those who earn ≥ 60k (true positives) is especially important.

The precision score of 66.59% suggests that when the model predicts someone earns ≥ 60k, it is correct some of the time, indicating some false positives. However, the recall score of 34.43% shows that the model missed a considerable number of true high earners (≥ 60k) capturing only about one third of them. This means that while the model is somewhat reliable in its positive predictions, it struggles to identify all individuals who truly belong in the ≥ 60k category.

### Do Race or Gender Skew Metrics of the Model?

#### Accuracy:

In [37]:
race_cat = lambda d: metrics.accuracy_score(merged_df['income>=60k'],
                       merged_df['LR_income>=60k'])
merged_df.groupby('race')[['income>=60k', 'LR_income>=60k']].apply(race_cat)

race
Amer-Indian-Eskimo    0.857027
Asian-Pac-Islander    0.857027
Black                 0.857027
Other                 0.857027
White                 0.857027
dtype: float64

In [38]:
gender_cat = lambda d: metrics.accuracy_score(merged_df['income>=60k'],
                       merged_df['LR_income>=60k'])
merged_df.groupby('gender')[['income>=60k', 'LR_income>=60k']].apply(gender_cat)

gender
Female    0.857027
Male      0.857027
dtype: float64

#### Precision:

In [41]:
race_cat = lambda d: metrics.precision_score(merged_df['income>=60k'],
                       merged_df['LR_income>=60k'])
merged_df.groupby('race')[['income>=60k', 'LR_income>=60k']].apply(race_cat)

race
Amer-Indian-Eskimo    0.665912
Asian-Pac-Islander    0.665912
Black                 0.665912
Other                 0.665912
White                 0.665912
dtype: float64

In [42]:
gender_cat = lambda d: metrics.precision_score(merged_df['income>=60k'],
                       merged_df['LR_income>=60k'])
merged_df.groupby('gender')[['income>=60k', 'LR_income>=60k']].apply(gender_cat)

gender
Female    0.665912
Male      0.665912
dtype: float64

#### Recall:

In [104]:
race_cat = lambda d: metrics.recall_score(merged_df['income>=60k'],
                       merged_df['LR_income>=60k'])
merged_df.groupby('race')[['income>=60k', 'LR_income>=60k']].apply(race_cat)

race
Amer-Indian-Eskimo    0.344262
Asian-Pac-Islander    0.344262
Black                 0.344262
Other                 0.344262
White                 0.344262
dtype: float64

In [106]:
gender_cat = lambda d: metrics.recall_score(merged_df['income>=60k'],
                       merged_df['LR_income>=60k'])
merged_df.groupby('gender')[['income>=60k', 'LR_income>=60k']].apply(gender_cat)

gender
Female    0.344262
Male      0.344262
dtype: float64

When splitting the accuracy, precision, and recall scores by race and gender, we can see that there is no variance between groups for any of the three metrics. This means that the model performs equally well across different racial and gender groups, suggesting that it is fair in its predictions and does not favor one group over another.

**********
### My Choice

Between the two models, the GPR model would be the better choice. Even though both models have similar overall performance, the GPR model performs slightly better across all three key metrics:

* Accuracy: 86.43% vs. 85.70% for LR
* Precision: 70.58% vs. 66.59% for LR
* Recall: 36.65% vs. 34.43% for LR

These differences may not seem large, but they indicate that the GPR model makes slightly more correct predictions overall, is more precise in identifying true high earners, and catches a few more of them compared to the LR model. Since the goal of the model is to accurately identify individuals who make ≥ 60k, recall and precision are more important. A model that balances both while maintaining good accuracy is more effective and fair. Therefore, the GPR model would be preferred because it achieves a better balance between accuracy, precision, and recall, meaning it is both more reliable and more capable of identifying high earners correctly when compared to LR.